# RSNA Knee Abnormality Detection

The code lives in the `rsna_knee` package under `src/`, not in this notebook.
One source of truth: the package is tested (`pytest`), and the notebook is
where you look at what it produced.

| module | what it does |
|---|---|
| `rsna_knee.dicom` | where the pixels are, and every header except the pixels |
| `rsna_knee.labeling` | free-text reports &rarr; 12 soft labels, in ~12 languages |
| `rsna_knee.records` | one series, pixels read on demand |
| `rsna_knee.builder` | the join, and the cached table it produces |


In [ ]:
# On Kaggle, point Python at the package (it is not pip-installed there).
import sys
sys.path.insert(0, "../src")        # or "/kaggle/working/<repo>/src"

# Locally:  pip install -e .


In [ ]:
import pandas as pd

from rsna_knee import KneeDatasetBuilder, resolve_data_path

DATA_PATH = resolve_data_path()     # $RSNA_DATA_PATH, the Kaggle mount, or ./data
DATA_PATH


## Build the index

One row per series: DICOM metadata + the study's report-derived labels.
The header scan is the slow part and runs once.


In [ ]:
builder = KneeDatasetBuilder(DATA_PATH, workers=8)
df = builder.build()
print(df.shape)
df.head()


## Check it before trusting it

Two failure modes that are silent if you do not look for them: series that
could not be read, and reports in a language the vocabulary does not cover.
The second is the dangerous one -- those rows come out all-negative for the
wrong reason, and training on them as negatives poisons the label set.


In [ ]:
print(len(builder.errors), "series could not be scanned")
for e in builder.errors[:5]:
    print(" ", e["SeriesInstanceUID"], e["scan_error"] or "missing folder")

display(builder.review_queue())
builder.labeler.coverage_report(zip(df["StudyInstanceUID"], df["report"]))


## What the corpus looks like

In [ ]:
display(df.groupby(["plane", "weighting"]).size().unstack(fill_value=0))

label_cols = builder.labels
rates = (df.drop_duplicates("StudyInstanceUID")[label_cols] > 0.5).mean()
rates.sort_values(ascending=False).to_frame("positive rate")


## Cache it

The scan only changes when the data does.


In [ ]:
builder.save("knee_index.parquet")
# next session:  builder = KneeDatasetBuilder.load("knee_index.parquet")


## Feed a model

`KneeDataset` duck-types as a `torch.utils.data.Dataset`, so it drops into a
`DataLoader` unchanged. Split on `group` (the study) -- every series of a study
shares one report, so a row-level split puts the same label on both sides.


In [ ]:
# Sagittal fat-suppressed series: the ones the menisci are actually read on.
ds = (builder.dataset(normalize="zscore")
             .filter(lambda r: r.plane == "sagittal" and r.meta["fat_saturated"]))
print(ds)

sample = ds[0]
print(sample["volume"].shape, sample["labels"].round(2))


In [ ]:
# from sklearn.model_selection import GroupShuffleSplit
# train_idx, val_idx = next(
#     GroupShuffleSplit(test_size=0.2, random_state=0).split(df, groups=df["group"]))
#
# from torch.utils.data import DataLoader
# loader = DataLoader(ds, batch_size=1, collate_fn=lambda b: b)   # ragged depths
